# Gestion de comptes bancaires

L'objectif de ce TP est d'implémenter un code qui gère des comptes bancaires. Ce TP vous demandera d'implémenter ce code deux fois : le premier en restant sur un paradigme procédural, puis la seconde implémentation ou chaque compte bancaire sera un objet.

## Spécifications
Voici les spécifications qui vous sont demandées de vérifier pour vos implémentations :
 - Quatre types de comptes sont possibles :
    - le compte courant
    - le compte de dépot
    - le livret A
    - le compte joint
 - Un compte est constitué d'un propriétaire et d'un solde
 - Un propriétaire peut avoir au plus un compte de chaque type
 - Un propriétaire peut déposer de l'argent ou en retirer (sauf cas particuliers. Voir plus bas)
 - Un propriétaire peut transférer de l'argent (avec les mêmes contraintes que pour le retrait ou le dépot) sur un de ses comptes ou d'un autre propriétaire

En fonction du type de compte, un certain nombre de contraintes sont à vérifier :
 - Un compte de dépot ne peut pas avoir un solde négatif
 - Un livret A ne peut pas avoir un solde supérieur à 29 500 €
 - Un compte joint doit avoir deux propriétaires

## Suite d'opérations
Le code suivant définit la fonction `suite_actions` qui va effectuer une série d'actions bancaires et quelques vérifications à l'issue de celles-ci.
Pour fonctionner, cette fonction nécessite l'implémentation de cinq fonctions.

Commencer par lire et comprendre le code ci-dessous.

In [ ]:


types_comptes = {"COMPTE_COURANT", "LIVRET_A", "COMPTE_DEPOT", "COMPTE_JOINT"}


def suite_actions() -> None:
  # Isabelle ouvre un compte courant et ajoute 1300 €
  cc_isabelle = creer_compte("COMPTE_COURANT", "isabelle")
  depot(cc_isabelle, "isabelle", 1300)

  # Jean ouvre un compte epargne et ajoute 1000 €
  cd_jean = creer_compte("COMPTE_DEPOT", "jean")
  depot(cd_jean, "jean", 1000)

  # Jean ouvre un livret A et ajoute 1200 €
  la_jean = creer_compte("LIVRET_A", "jean")
  depot(la_jean, "jean", 1200)

  # Isabelle et Jean ouvre un compte joint et dépose 6500 €
  cj_isabelle = creer_compte("COMPTE_JOINT", "isabelle", "jean")
  depot(cj_isabelle, "isabelle", 6500)

  # Isabelle transfert 500 € de son compte courant vers le compte joint
  transfert(cc_isabelle, cj_isabelle, "isabelle", 500)

  # Jean retire 60 e de son livret A
  retrait(la_jean, "jean", 60)

  # DÉBUT DES TESTS
  # Test des soldes des comptes
  assert solde(cc_isabelle, "isabelle") == 800
  assert solde(la_jean, "jean") == 1140
  assert solde(cj_isabelle, "isabelle") == 7000

  # Le compte ne peut pas avoir de solde négatif
  try:
    retrait(cd_jean, "jean", 2000)
  except ValueError:
    print("Solde insuffisant correctement détecté")
  else:
    print("Une erreur aurait dû se produire")

  # Le livret A ne peut pas avoir plus de 29500 €
  try:
    depot(la_jean, "jean", 29_000)
  except ValueError:
    print("Seuil max détecté")
  else:
    print("Une erreur aurait dû se produire")

## Implémentation impérative
La fonction `suite_actions` doit pouvoir s'exécuter normalement. Vous allez devoir implémenter les méthodes :
 - `creer_compte`
 - `depot`
 - `retrait`
 - `transfert`
 - `solde`

À vous d'implémenter ces fonctions. Vous pouvez ajouter toutes les fonctions ou structures de données que vous souhaitez, mais ne créez pas de classes pour le moment.

À la suite de ce code, la fonction `suite_actions` doit pouvoir être exécutée.

In [ ]:
from typing import Any


def creer_compte(
  type_compte: str, proprietaire: str, proprietaire2: str | None = None
) -> dict[str, Any]:
  # Votre code ici
  pass


def depot(compte: dict[str, Any], demandeur: str, montant: int) -> None:
  # Votre code ici
  pass


def retrait(compte: dict[str, Any], demandeur: str, montant: int) -> None:
  # Votre code ici
  pass


def transfert(
  compte_source: dict[str, Any],
  compte_cible: dict[str, Any],
  demandeur: str,
  montant: int,
) -> None:
  # Votre code ici
  pass


def solde(compte: dict[str, Any], demandeur: str) -> int:
  # Votre code ici
  pass

In [ ]:
# suite_actions()  # À faire tourner quand vous avez implémenté le code ci-dessus

### Solution

In [ ]:
from typing import Any


def creer_compte(
  type_compte: str, proprietaire: str, proprietaire2: str | None = None
) -> dict[str, Any]:
  assert (type_compte == "COMPTE_JOINT") != (proprietaire2 is None)
  return {
    "type": type_compte,
    "solde": 0,
    "proprietaire": proprietaire,
    "proprietaire2": proprietaire2,
  }


def depot(compte: dict[str, Any], demandeur: str, montant: int) -> None:
  _verifier_proprietaire(compte, demandeur)
  _verifier_conditions(compte, montant)
  compte["solde"] += montant


def retrait(compte: dict[str, Any], demandeur: str, montant: int) -> None:
  _verifier_proprietaire(compte, demandeur)
  _verifier_conditions(compte, -montant)
  compte["solde"] -= montant


def transfert(
  compte_source: dict[str, Any],
  compte_cible: dict[str, Any],
  demandeur: str,
  montant: int,
) -> None:
  _verifier_proprietaire(compte_source, demandeur)
  _verifier_conditions(compte_source, -montant)
  _verifier_conditions(compte_cible, montant)
  compte_source["solde"] -= montant
  compte_cible["solde"] += montant


def solde(compte: dict[str, Any], demandeur: str) -> int:
  _verifier_proprietaire(compte, demandeur)
  return compte["solde"]


def _verifier_proprietaire(compte: dict[str, Any], demandeur: str) -> None:
  if not _est_proprietaire(compte, demandeur):
    raise ValueError(f"{demandeur} n'est pas autorisé à agir sur le compte.")


def _verifier_conditions(compte: dict[str, Any], montant: int) -> None:
  if compte["type"] == "LIVRET_A" and compte["solde"] + montant > 29500:
    raise ValueError("Seuil maximal de 29 500 € atteint")
  if compte["type"] == "COMPTE_DEPOT" and compte["solde"] + montant < 0:
    raise ValueError("Solde insuffisant")


def _est_proprietaire(compte: dict[str, Any], personne: str) -> bool:
  return personne in {compte["proprietaire"], compte["proprietaire2"]}

In [ ]:
suite_actions()

## Implémentation objet

Même exercice que précédemment, mais vous devrez cette fois implémenter les différents types de compte bancaire en tant qu'objet.

Toutes les classes de compte que vous créerez devront hériter de la classe `Compte` que vous devez également définir.

In [ ]:
def suite_actions() -> None:
  # Isabelle ouvre un compte courant et ajoute 1300 €
  cc_isabelle = CompteCourant("isabelle")
  cc_isabelle.depot("isabelle", 1300)

  # Jean ouvre un compte epargne et ajoute 1000 €
  cd_jean = CompteDepot("jean")
  cd_jean.depot("jean", 1000)

  # Jean ouvre un livret A et ajoute 1200 €
  la_jean = LivretA("jean")
  la_jean.depot("jean", 1200)

  # Isabelle et Jean ouvre un compte joint et dépose 6500 €
  cj_isabelle = CompteJoint("isabelle", "jean")
  cj_isabelle.depot("isabelle", 6500)

  # Isabelle transfert 500 € de son compte courant vers le compte joint
  cc_isabelle.transfert("isabelle", cj_isabelle, 500)

  # Jean retire 60 e de son livret A
  la_jean.retrait("jean", 60)

  # DÉBUT DES TESTS
  # Test des soldes des comptes
  assert cc_isabelle.solde("isabelle") == 800
  assert la_jean.solde("jean") == 1140
  assert cj_isabelle.solde("isabelle") == 7000

  # Le compte ne peut pas avoir de solde négatif
  try:
    cd_jean.retrait("jean", 2000)
  except ValueError:
    print("Solde insuffisant correctement détecté")
  else:
    print("Une erreur aurait dû se produire")

  # Le livret A ne peut pas avoir plus de 29500 €
  try:
    la_jean.depot("jean", 29_000)
  except ValueError:
    print("Seuil max détecté")
  else:
    print("Une erreur aurait dû se produire")

In [ ]:
class Compte:
  def __init__(self, proprietaire: str):
    self._proprietaire = proprietaire
    self._solde = 0

  def solde(self, demandeur: str) -> int:
    # Votre code ici
    NotImplemented

  def depot(self, demandeur: str, montant: int) -> None:
    # Votre code ici
    NotImplemented

  def retrait(self, demandeur: str, montant: int) -> None:
    # Votre code ici
    NotImplemented

  def transfert(self, demandeur: str, other: "Compte", montant: int) -> None:
    # Votre code ici
    NotImplemented


class CompteCourant(Compte):
  # Votre code ici
  pass


class CompteDepot(Compte):
  # Votre code ici
  pass


class LivretA(Compte):
  # Votre code ici
  pass


class CompteJoint(Compte):
  # Votre code ici
  pass

In [ ]:
# suite_actions()  # À faire tourner quand vous avez implémenté le code ci-dessus

### Solution

In [ ]:
class Compte:
  def __init__(self, proprietaire: str) -> None:
    self._proprietaire = proprietaire
    self._solde = 0

  def _verifier_proprietaire(self, demandeur: str) -> None:
    if not self.est_proprietaire(demandeur):
      raise ValueError(f"{demandeur} n'est pas autorisé à agir sur le compte.")

  def _ajoute_solde(self, montant: int) -> None:
    self._solde += montant

  def depot(self, demandeur: str, montant: int) -> None:
    self._verifier_proprietaire(demandeur)
    self._ajoute_solde(montant)

  def retrait(self, demandeur: str, montant: int) -> None:
    self._verifier_proprietaire(demandeur)
    self._ajoute_solde(-montant)

  def solde(self, demandeur: str) -> int:
    self._verifier_proprietaire(demandeur)
    return self._solde

  def transfert(self, demandeur: str, autre: "Compte", montant: int) -> None:
    self._verifier_proprietaire(demandeur)
    self._ajoute_solde(-montant)
    autre._ajoute_solde(montant)

  def est_proprietaire(self, proprietaire: str) -> bool:
    return self._proprietaire == proprietaire


class CompteCourant(Compte):
  pass


class CompteDepot(Compte):
  def _ajoute_solde(self, montant: int) -> None:
    if self._solde + montant < 0:
      raise ValueError("Pas de découvert possible.")
    super()._ajoute_solde(montant)


class LivretA(Compte):
  def _ajoute_solde(self, montant: int) -> None:
    if self._solde + montant > 29500:
      raise ValueError("Plafond dépassé.")
    super()._ajoute_solde(montant)


class CompteJoint(Compte):
  def __init__(self, proprietaire: str, proprietaire2: str) -> None:
    super().__init__(proprietaire)
    self._proprietaire2 = proprietaire2

  def est_proprietaire(self, proprietaire: str) -> bool:
    return proprietaire in {self._proprietaire, self._proprietaire2}

In [ ]:
suite_actions()